# Relatorio Tecnico — Modelo CERBERUS

Este relatorio detalha a arquitetura tecnica e a fundamentacao matematica do Modelo CERBERUS, uma estrategia quantitativa de alocacao dinamica e protecao de capital. O nome e uma alusao ao cao guardiao da mitologia grega: enquanto o modelo opera sobre o "reino invisivel" dos estados de mercado (HMM), tres metricas distintas — as cabecas de Cerbero — vigiam o risco sistemico, a instabilidade e a propria incerteza do modelo.

## 1. O Reino de Hades: Hidden Markov Model (HMM) com GMM

### A teoria
O mercado e modelado como um processo estocastico com estados latentes $S_t \in \{0, 1, 2\}$, representando:

- Estado 0: Baixa volatilidade (tendencia).
- Estado 1: Media volatilidade (reversao a media).
- Estado 2: Alta volatilidade (crash/caos).

### Diferencial: Gaussian Mixture Models (GMM)
Ao contrario de HMMs simples que utilizam uma unica Gaussiana, o CERBERUS utiliza o GMMHMM. Isso permite que a probabilidade de emissao de cada estado seja uma combinacao de multiplas distribuicoes normais:

$$P(x_t | S_t = i) = \sum_{k=1}^{M} w_{i,k} \mathcal{N}(x_t | \mu_{i,k}, \Sigma_{i,k})$$

**Por que usar GMM?** Mercados financeiros apresentam "caudas grossas" (leptocurtose). O GMM permite capturar esses eventos extremos e a assimetria dos retornos sem que o modelo ignore os dados centrais, tornando a deteccao de regimes muito mais precisa.

## 2. As Tres Cabecas de Cerbero (Moduladores de Risco)

O modelo nao confia cegamente na previsao do estado. Tres metricas auditam o cenario para decidir a exposicao final.

### Cabeca I: Conectividade de Rede (Risco Sistemico)
Esta metrica modela a B3 como um grafo complexo para identificar o "efeito manada".

- O codigo: calcula-se a matriz de correlacao de Pearson entre os ativos em uma janela movel (ex: 21 dias).
- A teoria: aplica-se um threshold de 0.7. Se a correlacao entre dois ativos supera este valor, cria-se uma aresta no grafo.
- Metrica de alerta: monitora-se o maior autovalor ($\lambda_{max}$) da matriz. Quando $\lambda_{max}$ sobe bruscamente, um unico fator explica quase toda a variancia do mercado.
- Acao: se a densidade da rede (conexoes reais / conexoes possiveis) ultrapassa o percentil de 80%, o Cerbero corta a exposicao pela metade (50%), antecipando um colapso sistemico.

### Cabeca II: Volatilidade da Volatilidade (VoV)
A VoV e a "segunda derivada" do risco.

- A teoria: se $\sigma_t$ e a volatilidade anualizada dos precos, a VoV e o desvio padrao de $\sigma_t$.
- A logica: mudancas lentas na volatilidade sao aceitaveis. Picos na VoV indicam que o proprio regime de risco esta mudando de forma instavel.
- Acao: funciona como um acelerometro. Picos de VoV sinalizam que o HMM pode estar prestes a mudar de estado, servindo como aviso antecipado para reduzir posicoes defensivamente.

### Cabeca III: Entropia de Shannon (Incerteza do Modelo)
Aqui, o modelo pratica a "autocritica" usando a Teoria da Informacao.

- A teoria: o HMM gera um vetor de probabilidades $P = [p_0, p_1, p_2]$. A entropia de Shannon mede o grau de desordem ou incerteza desse vetor:

$$H = -\sum_{i=1}^{3} p_i \log_2(p_i)$$

- O limiar: a entropia maxima para 3 estados e $\approx 1.58$.
- Acao: no codigo, se Entropia > 1.2, o modelo assume que esta "confuso" (as probabilidades estao muito distribuidas, sem um vencedor claro). Nesse cenario de baixa conviccao, o Cerbero reduz o tamanho das novas posicoes pela metade para evitar sinais falsos.

## 3. Backtest e Regras de Execucao

O motor de backtest no arquivo DIPQ.py traduz a teoria em ordens de compra e venda.

### Filtro de Estado (Hades)
- Estado 0: ativa logica de cruzamento de medias moveis (SMA).
- Estado 1: ativa logica de RSI (IFR) para comprar ativos sobrevendidos.
- Estado 2: bloqueio total. O Cerbero fecha todas as posicoes e impede novas compras (protecao contra crashes).

### Gestao de Risco Implacavel
- Stop loss fixo (7% a 15%): limite maximo de perda por operacao.
- Trailing stop (5% a 12%): protege o lucro acumulado, subindo o stop conforme o preco sobe.
- CDI como benchmark: o capital nao alocado rende uma fracao do CDI, garantindo que o "caixa" tambem trabalhe para o portfolio.
